In [46]:
import json
from collections import defaultdict
import random
import torch

## 平均分为10份（10*150），每次取出其中一份（150）作为测试 + 随机取出150个

In [2]:
sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))

In [26]:
span = int(1500 / 10 / 5)

In [27]:
type_d = defaultdict(list)
for text, item in sub_test_sen_to_ids.items():
    type_d[item['type']].append(text)

In [29]:
# 10折
test_data_11 = [[] for _ in range(10+1)]
for i in range(10):
    for type, text_list in type_d.items():
        test_data_11[i].extend(text_list[i * span: (i + 1) * span])

In [31]:
# 最后整体平均取150个
all_text_list = []
for type, text_list in type_d.items():
    all_text_list.extend(text_list)

random.seed(20230918)
test_data_11[10] = random.sample(all_text_list, 150)

## No Enhanced Retrieval

In [38]:
sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))

In [39]:
model_name_list = ['align-base', 'clipseg-rd64-refined', 'clip-vit-base-patch32', 'groupvit-gcc-yfcc']
name = model_name_list[0]

In [42]:
text_features = torch.load('../encoder/data/raw_text_features_{}.pt'.format(name), map_location=torch.device('cpu'))

In [48]:
def get_text_index(text, sub_test_sen_to_ids):
    index = list(sub_test_sen_to_ids.keys()).index(text)
    return index

In [49]:
get_text_index(test_data_11[0][10], sub_test_sen_to_ids)

10

## Enhanced

In [53]:
sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))
text_to_enhanced = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

In [54]:
model_name_list = ['align-base', 'clipseg-rd64-refined', 'clip-vit-base-patch32', 'groupvit-gcc-yfcc']
name = model_name_list[0]

In [56]:
text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),
                                       map_location=torch.device('cpu'))

## Retrieval with checked

In [2]:
import json
import torch
from tqdm import tqdm
from collections import Counter, defaultdict

def get_text_index(text, sub_test_sen_to_ids):
    index = list(sub_test_sen_to_ids.keys()).index(text)
    return index

In [1]:
enhance = 'yes'
model_name_list = ['align-base', 'clipseg-rd64-refined', 'clip-vit-base-patch32', 'groupvit-gcc-yfcc']
cfg = {
    'topk_num': 10,
}

In [27]:
sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))

text_to_enhanced = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

text_enhance_to_flag = json.load(open('../enhance/data/text_enhance_to_flag.json', 'r', encoding='utf-8'))

topk_num = cfg['topk_num']

for name in model_name_list:
    fload_recall_list = []
    print(name)
    for flod_i, test_data in enumerate(test_data_11):
        test_data_index_list = [get_text_index(text, sub_test_sen_to_ids) for text in test_data]
        image_features = torch.load('../encoder/data/image_features_{}.pt'.format(name),
                                    map_location=torch.device('cpu'))
        image_features = torch.stack(image_features).view(1000, -1)

        text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),
                                   map_location=torch.device('cpu')) # list

        raw_text_features = torch.load('../encoder/data/raw_text_features_{}.pt'.format(name),
                                       map_location=torch.device('cpu'))
        raw_text_features /= raw_text_features.norm(dim=-1, keepdim=True)

        image_features /= image_features.norm(dim=-1, keepdim=True)

        recall_list = []
        for i, text in enumerate(test_data):
            enhance_list = text_to_enhanced[text]

            new_enhance_index_list = [n_i for n_i, enhance_text in enumerate(enhance_list) if text_enhance_to_flag[text][enhance_text] == 1] # 保留的enhance_text的索引

            type = sub_test_sen_to_ids[text]['type']
            index = get_text_index(text, sub_test_sen_to_ids)
            enhance_text_features = torch.cat([text_features[index][new_enhance_index_list,:], raw_text_features[index].view(1, -1)], 0)
            enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)
            similarity = (100.0 * enhance_text_features @ image_features.T).softmax(dim=-1)

            true_ids = sub_test_sen_to_ids[text]['images']
            true_ids_new = [img_id for img_id in true_ids if img_id in test_img_ids]
            true_ids = true_ids_new
            counter = Counter()
            for enhance_i in range(enhance_text_features.shape[0]):
                values, pre_indexes = similarity[enhance_i].topk(topk_num)
                pre_ids = [test_img_ids[j] for j in pre_indexes]
                counter.update(pre_ids)
            pre_ids = [img_id for (img_id, _) in counter.most_common(topk_num)]
            r = 0
            for true_id in true_ids:
                if true_id in pre_ids:
                    r += 1
            recall_list.append(r / min(len(true_ids), topk_num))

        recall = sum(recall_list) / len(recall_list)
        fload_recall_list.append(recall)
        print('\tflod [{flod_i}], recall [{recall}]'.format(
            flod_i=flod_i,
            model=name,
            recall=recall
        ))
    print('model [{model}], recall [{recall}]'.format(
        model=name,
        recall=sum(fload_recall_list) / len(fload_recall_list)
        ))

align-base
	flod [0], recall [0.7431428571428573]
	flod [1], recall [0.7129603174603176]
	flod [2], recall [0.708357142857143]
	flod [3], recall [0.7054153439153438]
	flod [4], recall [0.7167883597883599]
	flod [5], recall [0.7067777777777777]
	flod [6], recall [0.6293492063492063]
	flod [7], recall [0.7286481481481482]
	flod [8], recall [0.7384814814814814]
	flod [9], recall [0.7929312169312169]
	flod [10], recall [0.7113888888888888]
model [align-base], recall [0.7176582491582493]
clipseg-rd64-refined
	flod [0], recall [0.6560793650793653]
	flod [1], recall [0.6513809523809525]
	flod [2], recall [0.6693095238095239]
	flod [3], recall [0.6231825396825397]
	flod [4], recall [0.6825185185185183]
	flod [5], recall [0.6932222222222221]
	flod [6], recall [0.584579365079365]
	flod [7], recall [0.6390555555555555]
	flod [8], recall [0.6930899470899469]
	flod [9], recall [0.7074708994708995]
	flod [10], recall [0.6182407407407406]
model [clipseg-rd64-refined], recall [0.6561936026936026]
clip

In [ ]:
def retrieval_enhance_v2(cfg):
    '''
    不用Counter，用相似度的平均 + 原Query
    :param cfg:
    :return:
    '''
    sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
    test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
    test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))

    text_to_enhanced = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

    topk_num = cfg['topk_num']

    for name in model_name_list:
        fload_recall_list = []
        print(name)
        for flod_i, test_data in enumerate(test_data_11):
            test_data_index_list = [get_text_index(text, sub_test_sen_to_ids) for text in test_data]
            image_features = torch.load('../encoder/data/image_features_{}.pt'.format(name),
                                        map_location=torch.device('cpu'))
            image_features = torch.stack(image_features).view(1000, -1)

            text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),
                                       map_location=torch.device('cpu')) # list

            raw_text_features = torch.load('../encoder/data/raw_text_features_{}.pt'.format(name), map_location=torch.device('cpu'))
            raw_text_features /= raw_text_features.norm(dim=-1, keepdim=True)


            image_features /= image_features.norm(dim=-1, keepdim=True)

            recall_list = []
            for i, text in enumerate(test_data):
                enhance_list = text_to_enhanced[text]
                type = sub_test_sen_to_ids[text]['type']

                new_enhance_index_list = [n_i for n_i, enhance_text in enumerate(enhance_list) if text_enhance_to_flag[text][enhance_text] == 1] # 保留的enhance_text的索引

                index = get_text_index(text, sub_test_sen_to_ids)
                enhance_text_features = torch.cat([text_features[index][new_enhance_index_list], raw_text_features[index].view(1, -1)], 0)

                enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)
                similarity = (100.0 * enhance_text_features @ image_features.T).softmax(dim=-1)

                true_ids = sub_test_sen_to_ids[text]['images']
                true_ids_new = [img_id for img_id in true_ids if img_id in test_img_ids]
                true_ids = true_ids_new

                img_to_sim = defaultdict(list)
                for enhance_i in range(enhance_text_features.shape[0]):
                    if enhance_i != enhance_text_features.shape[0] - 1:
                        rate = 0.3
                    else:
                        rate = 0.7
                    values, pre_indexes = similarity[enhance_i].topk(topk_num)
                    for values, j in zip(values, pre_indexes):
                        pre_id = test_img_ids[j]
                        img_to_sim[pre_id].append(values.item() * rate)
                img_sim_sorted = sorted(img_to_sim.items(), key=lambda item: sum(item[1]) / len(item[1]), reverse=True)
                pre_ids = [pre_id for (pre_id, _) in img_sim_sorted[:topk_num]]
                r = 0
                for true_id in true_ids:
                    if true_id in pre_ids:
                        r += 1
                recall_list.append(r / min(len(true_ids), topk_num))

            recall = sum(recall_list) / len(recall_list)
            fload_recall_list.append(recall)
            print('\tflod [{flod_i}], recall [{recall}]'.format(
                flod_i=flod_i,
                model=name,
                recall=recall
            ))
        print('model [{model}], recall [{recall}]'.format(
            model=name,
            recall=sum(fload_recall_list) / len(fload_recall_list)
        ))

In [39]:
def retrieval_enhance_v3(cfg):
    '''

    :param cfg:
    :return:
    '''
    sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
    test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
    test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))

    text_to_enhanced = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

    text_enhance_to_flag = json.load(open('../enhance/data/text_enhance_to_flag.json', 'r', encoding='utf-8'))

    topk_num = cfg['topk_num']

    for name in model_name_list:
        fload_recall_list = []
        print(name)
        for flod_i, test_data in enumerate(test_data_11):
            test_data_index_list = [get_text_index(text, sub_test_sen_to_ids) for text in test_data]
            image_features = torch.load('../encoder/data/image_features_{}.pt'.format(name),
                                        map_location=torch.device('cpu'))
            image_features = torch.stack(image_features).view(1000, -1)

            text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),
                                       map_location=torch.device('cpu')) # list

            raw_text_features = torch.load('../encoder/data/raw_text_features_{}.pt'.format(name), map_location=torch.device('cpu'))
            raw_text_features /= raw_text_features.norm(dim=-1, keepdim=True)


            image_features /= image_features.norm(dim=-1, keepdim=True)

            recall_list = []
            for i, text in enumerate(test_data):
                enhance_list = text_to_enhanced[text]
                type = sub_test_sen_to_ids[text]['type']

                 # check
                # new_enhance_index_list = [n_i for n_i, enhance_text in enumerate(enhance_list) if text_enhance_to_flag[text][enhance_text] == 1] # 保留的enhance_text的索引
                #
                # index = get_text_index(text, sub_test_sen_to_ids)
                # enhance_text_features = torch.cat([text_features[index][new_enhance_index_list], raw_text_features[index].view(1, -1)], 0)

                # no check
                index = get_text_index(text, text_to_enhanced)
                enhance_text_features = torch.cat([text_features[index], raw_text_features[index].view(1, -1)], 0)

                enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)
                similarity = (100.0 * enhance_text_features @ image_features.T).softmax(dim=-1)

                true_ids = sub_test_sen_to_ids[text]['images']
                true_ids_new = [img_id for img_id in true_ids if img_id in test_img_ids]
                true_ids = true_ids_new

                img_to_sim = defaultdict(list)

                first_reacall_img_index = []
                for enhance_i in range(enhance_text_features.shape[0]):
                    values, pre_indexes = similarity[enhance_i].topk(topk_num * 2)
                    for values, j in zip(values, pre_indexes):
                        pre_id = test_img_ids[j]
                        first_reacall_img_index.append(j.item())
                first_reacall_img_index = list(set(first_reacall_img_index))

                sub_img_features = image_features[first_reacall_img_index]
                similarity = (100.0 * enhance_text_features @ sub_img_features.T).softmax(dim=-1)

                counter = Counter()
                for enhance_i in range(enhance_text_features.shape[0]):
                    values, pre_indexes = similarity[enhance_i].topk(topk_num + 5)
                    pre_ids = [test_img_ids[first_reacall_img_index[j]] for j in pre_indexes]
                    counter.update(pre_ids)
                pre_ids = [img_id for (img_id, _) in counter.most_common(topk_num)]

                r = 0
                for true_id in true_ids:
                    if true_id in pre_ids:
                        r += 1
                recall_list.append(r / min(len(true_ids), topk_num))

            recall = sum(recall_list) / len(recall_list)
            fload_recall_list.append(recall)
            print('\tflod [{flod_i}], recall [{recall}]'.format(
                flod_i=flod_i,
                model=name,
                recall=recall
            ))
        print('model [{model}], recall [{recall}]'.format(
            model=name,
            recall=sum(fload_recall_list) / len(fload_recall_list)
        ))

In [40]:
retrieval_enhance_v3(cfg)

align-base
	flod [0], recall [0.7323756613756612]
	flod [1], recall [0.7162936507936508]
	flod [2], recall [0.7319682539682538]
	flod [3], recall [0.6945714285714286]
	flod [4], recall [0.7308518518518518]
	flod [5], recall [0.7206666666666667]
	flod [6], recall [0.6491349206349207]
	flod [7], recall [0.7500185185185184]
	flod [8], recall [0.7524444444444445]
	flod [9], recall [0.780042328042328]
	flod [10], recall [0.736574074074074]
model [align-base], recall [0.7268128908128907]
clipseg-rd64-refined
	flod [0], recall [0.6525132275132274]
	flod [1], recall [0.6504444444444445]
	flod [2], recall [0.6816349206349209]
	flod [3], recall [0.6460158730158732]
	flod [4], recall [0.6816455026455026]
	flod [5], recall [0.6865555555555555]
	flod [6], recall [0.5845238095238096]
	flod [7], recall [0.6476666666666667]
	flod [8], recall [0.6969126984126984]
	flod [9], recall [0.7293597883597884]
	flod [10], recall [0.6758518518518518]
model [clipseg-rd64-refined], recall [0.6666476671476672]
clip

## Retrieval with sub check

In [1]:
import json
import torch
from tqdm import tqdm
from collections import Counter, defaultdict

def get_text_index(text, sub_test_sen_to_ids):
    index = list(sub_test_sen_to_ids.keys()).index(text)
    return index

In [2]:
enhance = 'yes'
model_name_list = ['align-base', 'clipseg-rd64-refined', 'clip-vit-base-patch32', 'groupvit-gcc-yfcc']
cfg = {
    'topk_num': 10,
}

In [13]:
sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))

text_to_enhanced = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

text_enhance_to_flag = json.load(open('/Users/aidan/Learn/NLP/Retrieval/Construction/V8/enhance/chatGPT/data/text_enhance_to_flag_sub.json', 'r', encoding='utf-8'))

topk_num = cfg['topk_num']

for name in model_name_list:
    fload_recall_list = []
    print(name)
    for flod_i, test_data in enumerate(test_data_11):
        if flod_i in [4,10]:
            test_data_index_list = [get_text_index(text, sub_test_sen_to_ids) for text in test_data]
            image_features = torch.load('../encoder/data/image_features_{}.pt'.format(name),
                                        map_location=torch.device('cpu'))
            image_features = torch.stack(image_features).view(1000, -1)

            text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),
                                       map_location=torch.device('cpu')) # list

            raw_text_features = torch.load('../encoder/data/raw_text_features_{}.pt'.format(name),
                                           map_location=torch.device('cpu'))
            raw_text_features /= raw_text_features.norm(dim=-1, keepdim=True)

            image_features /= image_features.norm(dim=-1, keepdim=True)

            recall_list = []
            for i, text in enumerate(test_data):
                enhance_list = text_to_enhanced[text]

                new_enhance_index_list = [n_i for n_i, enhance_text in enumerate(enhance_list) if text_enhance_to_flag[text][enhance_text] == 1] # 保留的enhance_text的索引

                type = sub_test_sen_to_ids[text]['type']
                index = get_text_index(text, sub_test_sen_to_ids)
                enhance_text_features = torch.cat([text_features[index][new_enhance_index_list,:], raw_text_features[index].view(1, -1)], 0)
                enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)
                similarity = (100.0 * enhance_text_features @ image_features.T).softmax(dim=-1)

                true_ids = sub_test_sen_to_ids[text]['images']
                true_ids_new = [img_id for img_id in true_ids if img_id in test_img_ids]
                true_ids = true_ids_new
                first_reacall_img_index = []
                for enhance_i in range(enhance_text_features.shape[0]):
                    values, pre_indexes = similarity[enhance_i].topk(topk_num * 2)
                    for values, j in zip(values, pre_indexes):
                        pre_id = test_img_ids[j]
                        first_reacall_img_index.append(j.item())
                first_reacall_img_index = list(set(first_reacall_img_index))

                sub_img_features = image_features[first_reacall_img_index]
                similarity = (100.0 * enhance_text_features @ sub_img_features.T).softmax(dim=-1)

                counter = Counter()
                for enhance_i in range(enhance_text_features.shape[0]):
                    values, pre_indexes = similarity[enhance_i].topk(topk_num)
                    pre_ids = [test_img_ids[first_reacall_img_index[j]] for j in pre_indexes]
                    counter.update(pre_ids)
                pre_ids = [img_id for (img_id, _) in counter.most_common(topk_num)]
                r = 0
                for true_id in true_ids:
                    if true_id in pre_ids:
                        r += 1
                recall_list.append(r / min(len(true_ids), topk_num))

            recall = sum(recall_list) / len(recall_list)
            fload_recall_list.append(recall)
            print('\tflod [{flod_i}], recall [{recall}]'.format(
                flod_i=flod_i,
                model=name,
                recall=recall
            ))
    print('model [{model}], recall [{recall}]'.format(
        model=name,
        recall=sum(fload_recall_list) / len(fload_recall_list)
        ))

align-base
	flod [4], recall [0.7117883597883596]
	flod [10], recall [0.7045740740740741]
model [align-base], recall [0.7081812169312169]
clipseg-rd64-refined
	flod [4], recall [0.6888412698412698]
	flod [10], recall [0.6540185185185186]
model [clipseg-rd64-refined], recall [0.6714298941798942]
clip-vit-base-patch32
	flod [4], recall [0.646973544973545]
	flod [10], recall [0.6404629629629629]
model [clip-vit-base-patch32], recall [0.643718253968254]
groupvit-gcc-yfcc
	flod [4], recall [0.5762380952380952]
	flod [10], recall [0.5393888888888889]
model [groupvit-gcc-yfcc], recall [0.5578134920634921]


## Chatgpt

In [32]:


def get_text_index(text, text_to_enhanced):
    index = list(text_to_enhanced.keys()).index(text)
    return index


sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))

text_to_enhanced = json.load(open('../enhance/chatGPT/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

text_enhance_to_flag = json.load(open('../enhance/chatGPT/data/text_enhance_to_flag_sub.json', 'r', encoding='utf-8'))

topk_num = cfg['topk_num']

for name in model_name_list:
    fload_recall_list = []
    print(name)
    for flod_i, test_data in enumerate(test_data_11):
        if flod_i in [2,7]:
            test_data_index_list = [get_text_index(text, sub_test_sen_to_ids) for text in test_data]
            image_features = torch.load('../encoder/data/image_features_{}.pt'.format(name),
                                        map_location=torch.device('cpu'))
            image_features = torch.stack(image_features).view(1000, -1)

            text_features = torch.load('../enhance/chatGPT/data/enhanced_text_features_{}.pt'.format(name),map_location=torch.device('cpu')) # list

            raw_text_features = torch.load('../encoder/data/raw_text_features_{}.pt'.format(name), map_location=torch.device('cpu'))
            raw_text_features /= raw_text_features.norm(dim=-1, keepdim=True)


            image_features /= image_features.norm(dim=-1, keepdim=True)

            recall_list = []
            for i, text in enumerate(test_data):
                enhance_list = text_to_enhanced[text]
                type = sub_test_sen_to_ids[text]['type']

                # check
                # new_enhance_index_list = [n_i for n_i, enhance_text in enumerate(enhance_list) if text_enhance_to_flag[text][enhance_text] == 1] # 保留的enhance_text的索引
                #
                # enhance_index = get_text_index(text, text_to_enhanced)
                # raw_index = get_text_index(text, sub_test_sen_to_ids)
                # enhance_text_features = torch.cat([text_features[enhance_index][new_enhance_index_list], raw_text_features[raw_index].view(1, -1)], 0)
                # enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)

                # no check
                enhance_index = get_text_index(text, text_to_enhanced)
                raw_index = get_text_index(text, sub_test_sen_to_ids)
                enhance_text_features = torch.cat([text_features[enhance_index], raw_text_features[raw_index].view(1, -1)], 0)
                enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)


                # if type in ['fragment', 'phrase']:
                #     index = get_text_index(text, text_to_enhanced)
                #     enhance_text_features = torch.cat([text_features[index], raw_text_features[index].view(1, -1)], 0)
                #     enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)
                # else:
                #    enhance_text_features = raw_text_features[index].view(1, -1)
                #    # enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)

                similarity = (100.0 * enhance_text_features @ image_features.T).softmax(dim=-1)

                true_ids = sub_test_sen_to_ids[text]['images']
                true_ids_new = [img_id for img_id in true_ids if img_id in test_img_ids]
                true_ids = true_ids_new

                img_to_sim = defaultdict(list)

                first_reacall_img_index = []
                for enhance_i in range(enhance_text_features.shape[0]):
                    values, pre_indexes = similarity[enhance_i].topk(topk_num * 2)
                    for values, j in zip(values, pre_indexes):
                        pre_id = test_img_ids[j]
                        first_reacall_img_index.append(j.item())
                first_reacall_img_index = list(set(first_reacall_img_index))

                sub_img_features = image_features[first_reacall_img_index]
                similarity = (100.0 * enhance_text_features @ sub_img_features.T).softmax(dim=-1)

                counter = Counter()
                for enhance_i in range(enhance_text_features.shape[0]):
                    values, pre_indexes = similarity[enhance_i].topk(topk_num + 5)
                    pre_ids = [test_img_ids[first_reacall_img_index[j]] for j in pre_indexes]
                    counter.update(pre_ids)
                pre_ids = [img_id for (img_id, _) in counter.most_common(topk_num)]

                r = 0
                for true_id in true_ids:
                    if true_id in pre_ids:
                        r += 1
                recall_list.append(r / min(len(true_ids), topk_num))

            recall = sum(recall_list) / len(recall_list)
            fload_recall_list.append(recall)
            print('\tflod [{flod_i}], recall [{recall}]'.format(
                flod_i=flod_i,
                model=name,
                recall=recall
            ))
    print('model [{model}], recall [{recall}]'.format(
        model=name,
        recall=sum(fload_recall_list) / len(fload_recall_list)
    ))

align-base
	flod [2], recall [0.7397857142857144]
	flod [7], recall [0.7339074074074073]
model [align-base], recall [0.7368465608465609]
clipseg-rd64-refined
	flod [2], recall [0.7091349206349209]
	flod [7], recall [0.6651111111111111]
model [clipseg-rd64-refined], recall [0.6871230158730159]
clip-vit-base-patch32
	flod [2], recall [0.6166111111111111]
	flod [7], recall [0.6846111111111112]
model [clip-vit-base-patch32], recall [0.6506111111111111]
groupvit-gcc-yfcc
	flod [2], recall [0.5350634920634921]
	flod [7], recall [0.5491481481481482]
model [groupvit-gcc-yfcc], recall [0.5421058201058202]


## retrieval with vote

In [1]:
import json
import torch
from tqdm import tqdm
from collections import Counter, defaultdict

In [2]:
enhance = 'yes'
model_name_list = ['align-base', 'clipseg-rd64-refined', 'clip-vit-base-patch32', 'groupvit-gcc-yfcc']
cfg = {
    'topk_num': 10,
}

In [3]:
import random

In [4]:
def get_text_index(text, text_to_enhanced):
    index = list(text_to_enhanced.keys()).index(text)
    return index

In [28]:

sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))

# text_to_enhanced = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))
# text_to_enhanced_2 = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

text_enhance_to_flag = json.load(open('../enhance/data/text_enhance_to_flag.json', 'r', encoding='utf-8'))

topk_num = cfg['topk_num']

for name in model_name_list:

    text_to_enhanced = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))
    text_to_enhanced_2 = json.load(open('../enhance/data/text_to_enhanced_list_2.json', 'r', encoding='utf-8'))

    enhanced_text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),map_location=torch.device('cpu')) # list
    enhanced_text_features_2 = torch.load('../encoder/data/enhanced_text_features_{}_2.pt'.format(name),map_location=torch.device('cpu')) # list

    text_to_all_enhanced = defaultdict(list)
    enhanced_text_to_features_index = {}
    enhanced_all_text_features = []

    features_index = 0
    for i, text in enumerate(list(sub_test_sen_to_ids.keys())):
        enhanced_list = text_to_enhanced[text]
        features = enhanced_text_features[i]
        # if len(enhanced_list) == 0:
        #     print(text)
        for enhanced_text, feature in zip(enhanced_list, features):
            text_to_all_enhanced[text].append(enhanced_text)
            enhanced_text_to_features_index[enhanced_text] = features_index
            features_index += 1
            enhanced_all_text_features.append(feature)
        enhanced_list = text_to_enhanced_2[text]
        features = enhanced_text_features_2[i]

        for enhanced_text, feature in zip(enhanced_list, features):
            text_to_all_enhanced[text].append(enhanced_text)
            enhanced_text_to_features_index[enhanced_text] = features_index
            features_index += 1
            enhanced_all_text_features.append(feature)
    enhanced_all_text_features = torch.stack(enhanced_all_text_features)
    fload_recall_list = []
    print(name)
    type_to_index = {
        'rawSentence': 0,
        'simSentence': 1,
        'fragment': 2,
        'phrase':3,
        'tag':4
    }
    type_fload_recall_list = []
    for flod_i, test_data in enumerate(test_data_11):
        test_data_index_list = [get_text_index(text, sub_test_sen_to_ids) for text in test_data]
        image_features = torch.load('../encoder/data/image_features_{}.pt'.format(name),
                                    map_location=torch.device('cpu'))
        image_features = torch.stack(image_features).view(1000, -1)

        # text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),
        #                            map_location=torch.device('cpu')) # list
        # enhanced_text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),map_location=torch.device('cpu')) # list
        # enhanced_text_features_2 = torch.load('../encoder/data/enhanced_text_features_{}_2.pt'.format(name),map_location=torch.device('cpu')) # list

        raw_text_features = torch.load('../encoder/data/raw_text_features_{}.pt'.format(name), map_location=torch.device('cpu'))
        raw_text_features /= raw_text_features.norm(dim=-1, keepdim=True)


        image_features /= image_features.norm(dim=-1, keepdim=True)


        recall_list = []
        type_recall_list = [[] for _ in range(len(type_to_index))]
        for i, text in enumerate(test_data):
            enhanced_list = text_to_all_enhanced[text]
            # enhanced_index_list = [enhanced_text_to_features_index[enhanced_text] for enhanced_text in enhanced_list]
            # enhanced_text_features = enhanced_all_text_features[enhanced_index_list]
            type = sub_test_sen_to_ids[text]['type']
            index  = get_text_index(text, sub_test_sen_to_ids)

            true_ids = sub_test_sen_to_ids[text]['images']
            true_ids_new = [img_id for img_id in true_ids if img_id in test_img_ids]
            true_ids = true_ids_new


            if len(enhanced_list) <= 10:
                # group_num = int(len(enhanced_list)//3)
                similarity = (100.0 * raw_text_features[index] @ image_features.T).softmax(dim=-1)
                values, pre_indexes = similarity.topk(topk_num)
                pre_ids = [test_img_ids[j] for j in pre_indexes]
            else:
                group_num = 7
                counter = Counter()
                for u in range(0, len(enhanced_list), group_num):
                    sub_enhanced_list = enhanced_list[u: u+group_num]
                    sub_enhanced_index_list = [enhanced_text_to_features_index[enhanced_text] for enhanced_text in sub_enhanced_list]
                    sub_enhanced_text_features = enhanced_all_text_features[sub_enhanced_index_list]

                    sub_enhanced_text_features = torch.cat([sub_enhanced_text_features, raw_text_features[index].view(1, -1)], 0)

                    sub_enhanced_text_features /= sub_enhanced_text_features.norm(dim=-1, keepdim=True)
                    similarity = (100.0 * sub_enhanced_text_features @ image_features.T).softmax(dim=-1)

                    first_reacall_img_index = []
                    for enhance_i in range(sub_enhanced_text_features.shape[0]):
                        values, pre_indexes = similarity[enhance_i].topk(topk_num+5)
                        for values, j in zip(values, pre_indexes):
                            pre_id = test_img_ids[j]
                            first_reacall_img_index.append(j.item())
                    first_reacall_img_index = list(set(first_reacall_img_index))

                    sub_img_features = image_features[first_reacall_img_index]
                    similarity = (100.0 * sub_enhanced_text_features @ sub_img_features.T).softmax(dim=-1)

                    sub_counter = Counter()
                    for enhance_i in range(sub_enhanced_text_features.shape[0]):
                        values, pre_indexes = similarity[enhance_i].topk(topk_num+3)
                        # values, pre_indexes = similarity[enhance_i].topk(len(first_reacall_img_index))
                        pre_ids = [test_img_ids[first_reacall_img_index[j]] for j in pre_indexes]
                        sub_counter.update(pre_ids)
                    pre_ids = [img_id for (img_id, _) in sub_counter.most_common(topk_num+2)]
                    counter.update(pre_ids)
                pre_ids = [img_id for (img_id, _) in counter.most_common(topk_num)]
            r = 0
            for true_id in true_ids:
                if true_id in pre_ids:
                    r += 1
            recall_list.append(r / min(len(true_ids), topk_num))
            type_recall_list[type_to_index[type]].append(r / min(len(true_ids), topk_num))
            # check
            # new_enhance_index_list = [n_i for n_i, enhance_text in enumerate(enhance_list) if text_enhance_to_flag[text][enhance_text] == 1] # 保留的enhance_text的索引
            #
            # index = get_text_index(text, sub_test_sen_to_ids)
            # enhance_text_features = torch.cat([text_features[index][new_enhance_index_list], raw_text_features[index].view(1, -1)], 0)
        recall = sum(recall_list) / len(recall_list)
        type_recall = [sum(r)/len(r) for r in type_recall_list]
        type_fload_recall_list.append(type_recall)
        fload_recall_list.append(recall)
        print('\tflod [{flod_i}], recall [{recall}]'.format(
            flod_i=flod_i,
            model=name,
            recall=recall
        ))
    final_type_recall = [0 for _ in range(len(type_to_index))]
    for type_recall in type_fload_recall_list:
        for i, r in enumerate(type_recall):
            final_type_recall[i] += r
    final_type_recall = [i/len(type_fload_recall_list) for i in final_type_recall]
    print('model [{model}], recall [{recall}], type {final_type_recall}'.format(
        model=name,
        recall=sum(fload_recall_list) / len(fload_recall_list),
        final_type_recall = final_type_recall
    ))

align-base
	flod [0], recall [0.7266613756613756]
	flod [1], recall [0.7137936507936509]
	flod [2], recall [0.7437539682539682]
	flod [3], recall [0.6995714285714287]
	flod [4], recall [0.7315661375661373]
	flod [5], recall [0.7166666666666667]
	flod [6], recall [0.6366904761904763]
	flod [7], recall [0.7379814814814815]
	flod [8], recall [0.760563492063492]
	flod [9], recall [0.7919312169312168]
	flod [10], recall [0.7406851851851851]
model [align-base], recall [0.7272604617604619], type [0.9787878787878789, 0.8549364613880743, 0.8259740259740261, 0.5934995112414467, 0.38327200577200576]
clipseg-rd64-refined
	flod [0], recall [0.6568095238095238]
	flod [1], recall [0.659277777777778]
	flod [2], recall [0.6919126984126986]
	flod [3], recall [0.6347301587301587]
	flod [4], recall [0.6971375661375661]
	flod [5], recall [0.6865555555555555]
	flod [6], recall [0.5906349206349206]
	flod [7], recall [0.6352037037037038]
	flod [8], recall [0.6897936507936506]
	flod [9], recall [0.733415343915

## 原Metric

In [6]:

sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))

# text_to_enhanced = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))
# text_to_enhanced_2 = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

text_enhance_to_flag = json.load(open('../enhance/data/text_enhance_to_flag.json', 'r', encoding='utf-8'))

topk_num = cfg['topk_num']

for name in model_name_list:

    text_to_enhanced = json.load(open('../enhance/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))
    text_to_enhanced_2 = json.load(open('../enhance/data/text_to_enhanced_list_2.json', 'r', encoding='utf-8'))

    enhanced_text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),map_location=torch.device('cpu')) # list
    enhanced_text_features_2 = torch.load('../encoder/data/enhanced_text_features_{}_2.pt'.format(name),map_location=torch.device('cpu')) # list

    text_to_all_enhanced = defaultdict(list)
    enhanced_text_to_features_index = {}
    enhanced_all_text_features = []

    features_index = 0
    for i, text in enumerate(list(sub_test_sen_to_ids.keys())):
        enhanced_list = text_to_enhanced[text]
        features = enhanced_text_features[i]
        # if len(enhanced_list) == 0:
        #     print(text)
        for enhanced_text, feature in zip(enhanced_list, features):
            text_to_all_enhanced[text].append(enhanced_text)
            enhanced_text_to_features_index[enhanced_text] = features_index
            features_index += 1
            enhanced_all_text_features.append(feature)
        enhanced_list = text_to_enhanced_2[text]
        features = enhanced_text_features_2[i]

        for enhanced_text, feature in zip(enhanced_list, features):
            text_to_all_enhanced[text].append(enhanced_text)
            enhanced_text_to_features_index[enhanced_text] = features_index
            features_index += 1
            enhanced_all_text_features.append(feature)
    enhanced_all_text_features = torch.stack(enhanced_all_text_features)
    fload_recall_list = []
    print(name)
    type_to_index = {
        'rawSentence': 0,
        'simSentence': 1,
        'fragment': 2,
        'phrase':3,
        'tag':4
    }
    type_fload_recall_list = []
    for flod_i, test_data in enumerate(test_data_11):
        test_data_index_list = [get_text_index(text, sub_test_sen_to_ids) for text in test_data]
        image_features = torch.load('../encoder/data/image_features_{}.pt'.format(name),
                                    map_location=torch.device('cpu'))
        image_features = torch.stack(image_features).view(1000, -1)

        # text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),
        #                            map_location=torch.device('cpu')) # list
        # enhanced_text_features = torch.load('../encoder/data/enhanced_text_features_{}.pt'.format(name),map_location=torch.device('cpu')) # list
        # enhanced_text_features_2 = torch.load('../encoder/data/enhanced_text_features_{}_2.pt'.format(name),map_location=torch.device('cpu')) # list

        raw_text_features = torch.load('../encoder/data/raw_text_features_{}.pt'.format(name), map_location=torch.device('cpu'))
        raw_text_features /= raw_text_features.norm(dim=-1, keepdim=True)


        image_features /= image_features.norm(dim=-1, keepdim=True)


        recall_list = []
        type_recall_list = [[] for _ in range(len(type_to_index))]
        for i, text in enumerate(test_data):
            enhanced_list = text_to_all_enhanced[text]
            # enhanced_index_list = [enhanced_text_to_features_index[enhanced_text] for enhanced_text in enhanced_list]
            # enhanced_text_features = enhanced_all_text_features[enhanced_index_list]
            type = sub_test_sen_to_ids[text]['type']
            index  = get_text_index(text, sub_test_sen_to_ids)

            true_ids = sub_test_sen_to_ids[text]['images']
            true_ids_new = [img_id for img_id in true_ids if img_id in test_img_ids]
            true_ids = true_ids_new


            if len(enhanced_list) <= 10:
                # group_num = int(len(enhanced_list)//3)
                similarity = (100.0 * raw_text_features[index] @ image_features.T).softmax(dim=-1)
                values, pre_indexes = similarity.topk(topk_num)
                pre_ids = [test_img_ids[j] for j in pre_indexes]
            else:
                group_num = 7
                counter = Counter()


                for u in range(0, len(enhanced_list), group_num):
                    sub_enhanced_list = enhanced_list[u: u+group_num]
                    sub_enhanced_index_list = [enhanced_text_to_features_index[enhanced_text] for enhanced_text in sub_enhanced_list]
                    sub_enhanced_text_features = enhanced_all_text_features[sub_enhanced_index_list]

                    sub_enhanced_text_features = torch.cat([sub_enhanced_text_features, raw_text_features[index].view(1, -1)], 0)

                    sub_enhanced_text_features /= sub_enhanced_text_features.norm(dim=-1, keepdim=True)
                    similarity = (100.0 * sub_enhanced_text_features @ image_features.T).softmax(dim=-1)

                    first_reacall_img_index = []
                    for enhance_i in range(sub_enhanced_text_features.shape[0]):
                        values, pre_indexes = similarity[enhance_i].topk(topk_num+5)
                        for values, j in zip(values, pre_indexes):
                            pre_id = test_img_ids[j]
                            first_reacall_img_index.append(j.item())
                    first_reacall_img_index = list(set(first_reacall_img_index))

                    sub_img_features = image_features[first_reacall_img_index]
                    similarity = (100.0 * sub_enhanced_text_features @ sub_img_features.T).softmax(dim=-1)


                    sub_counter = Counter()
                    for enhance_i in range(sub_enhanced_text_features.shape[0]):
                        values, pre_indexes = similarity[enhance_i].topk(topk_num+3)
                        # values, pre_indexes = similarity[enhance_i].topk(len(first_reacall_img_index))
                        pre_ids = [test_img_ids[first_reacall_img_index[j]] for j in pre_indexes]
                        sub_counter.update(pre_ids)
                    pre_ids = [img_id for (img_id, _) in sub_counter.most_common(topk_num+2)]
                    counter.update(pre_ids)
                pre_ids = [img_id for (img_id, _) in counter.most_common(topk_num)]
            r = 0
            for true_id in true_ids:
                if true_id in pre_ids:
                    r = 1
                    break
            recall_list.append(r)
            type_recall_list[type_to_index[type]].append(r / min(len(true_ids), topk_num))
            # check
            # new_enhance_index_list = [n_i for n_i, enhance_text in enumerate(enhance_list) if text_enhance_to_flag[text][enhance_text] == 1] # 保留的enhance_text的索引
            #
            # index = get_text_index(text, sub_test_sen_to_ids)
            # enhance_text_features = torch.cat([text_features[index][new_enhance_index_list], raw_text_features[index].view(1, -1)], 0)
        recall = sum(recall_list) / len(recall_list)
        type_recall = [sum(r)/len(r) for r in type_recall_list]
        type_fload_recall_list.append(type_recall)
        fload_recall_list.append(recall)
        print('\tflod [{flod_i}], recall [{recall}]'.format(
            flod_i=flod_i,
            model=name,
            recall=recall
        ))
    final_type_recall = [0 for _ in range(len(type_to_index))]
    for type_recall in type_fload_recall_list:
        for i, r in enumerate(type_recall):
            final_type_recall[i] += r
    final_type_recall = [i/len(type_fload_recall_list) for i in final_type_recall]
    print('model [{model}], recall [{recall}], type {final_type_recall}'.format(
        model=name,
        recall=sum(fload_recall_list) / len(fload_recall_list),
        final_type_recall = final_type_recall
    ))

align-base
	flod [0], recall [0.82]
	flod [1], recall [0.7866666666666666]
	flod [2], recall [0.82]
	flod [3], recall [0.7666666666666667]
	flod [4], recall [0.8133333333333334]
	flod [5], recall [0.7866666666666666]
	flod [6], recall [0.7066666666666667]
	flod [7], recall [0.8]
	flod [8], recall [0.8466666666666667]
	flod [9], recall [0.8866666666666667]
	flod [10], recall [0.84]
model [align-base], recall [0.8066666666666666], type [0.9772727272727273, 0.8549364613880743, 0.8244588744588746, 0.5746252851091561, 0.19874579124579125]
clipseg-rd64-refined
	flod [0], recall [0.7533333333333333]
	flod [1], recall [0.74]
	flod [2], recall [0.7533333333333333]
	flod [3], recall [0.6933333333333334]
	flod [4], recall [0.7666666666666667]
	flod [5], recall [0.76]
	flod [6], recall [0.6533333333333333]
	flod [7], recall [0.7]
	flod [8], recall [0.7666666666666667]
	flod [9], recall [0.8133333333333334]
	flod [10], recall [0.7666666666666667]
model [clipseg-rd64-refined], recall [0.742424242424

Chatgpt with vote

In [43]:


def get_text_index(text, text_to_enhanced):
    index = list(text_to_enhanced.keys()).index(text)
    return index


sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))

text_to_enhanced = json.load(open('../enhance/chatGPT/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

text_enhance_to_flag = json.load(open('../enhance/chatGPT/data/text_enhance_to_flag_sub.json', 'r', encoding='utf-8'))

topk_num = cfg['topk_num']

for name in model_name_list:
    fload_recall_list = []

    text_to_enhanced = json.load(open('../enhance/chatGPT/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))
    text_to_enhanced_2 = json.load(open('../enhance/chatGPT/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

    enhanced_text_features = torch.load('../enhance/chatGPT/data/enhanced_text_features_{}.pt'.format(name),map_location=torch.device('cpu')) # list
    enhanced_text_features_2 = torch.load('../enhance/chatGPT/data/enhanced_text_features_{}_2.pt'.format(name),map_location=torch.device('cpu')) # list

    text_to_all_enhanced = defaultdict(list)
    enhanced_text_to_features_index = {}
    enhanced_all_text_features = []

    features_index = 0
    for i, text in enumerate(test_data_11[2] + test_data_11[7]):
        enhanced_list = text_to_enhanced[text]
        features = enhanced_text_features[i]
        # if len(enhanced_list) == 0:
        #     print(text)
        for enhanced_text, feature in zip(enhanced_list, features):
            text_to_all_enhanced[text].append(enhanced_text)
            enhanced_text_to_features_index[enhanced_text] = features_index
            features_index += 1
            enhanced_all_text_features.append(feature)
        enhanced_list = text_to_enhanced_2[text]
        features = enhanced_text_features_2[i]

        for enhanced_text, feature in zip(enhanced_list, features):
            text_to_all_enhanced[text].append(enhanced_text)
            enhanced_text_to_features_index[enhanced_text] = features_index
            features_index += 1
            enhanced_all_text_features.append(feature)
    enhanced_all_text_features = torch.stack(enhanced_all_text_features)
    print(name)
    for flod_i, test_data in enumerate(test_data_11):
        if flod_i in [2,7]:
            test_data_index_list = [get_text_index(text, sub_test_sen_to_ids) for text in test_data]
            image_features = torch.load('../encoder/data/image_features_{}.pt'.format(name),
                                        map_location=torch.device('cpu'))
            image_features = torch.stack(image_features).view(1000, -1)

            text_features = torch.load('../enhance/chatGPT/data/enhanced_text_features_{}.pt'.format(name),map_location=torch.device('cpu')) # list

            raw_text_features = torch.load('../encoder/data/raw_text_features_{}.pt'.format(name), map_location=torch.device('cpu'))
            raw_text_features /= raw_text_features.norm(dim=-1, keepdim=True)


            image_features /= image_features.norm(dim=-1, keepdim=True)

            recall_list = []
            for i, text in enumerate(test_data):
                enhanced_list = text_to_all_enhanced[text]
                # enhanced_index_list = [enhanced_text_to_features_index[enhanced_text] for enhanced_text in enhanced_list]
                # enhanced_text_features = enhanced_all_text_features[enhanced_index_list]
                type = sub_test_sen_to_ids[text]['type']
                index  = get_text_index(text, sub_test_sen_to_ids)

                true_ids = sub_test_sen_to_ids[text]['images']
                true_ids_new = [img_id for img_id in true_ids if img_id in test_img_ids]
                true_ids = true_ids_new


                if len(enhanced_list) <= 0:
                    # group_num = int(len(enhanced_list)//3)
                    similarity = (100.0 * raw_text_features[index] @ image_features.T).softmax(dim=-1)
                    values, pre_indexes = similarity.topk(topk_num)
                    pre_ids = [test_img_ids[j] for j in pre_indexes]
                else:
                    group_num = 4 #4-10-8-5,
                    counter = Counter()

                    for u in range(0, len(enhanced_list), group_num):
                        sub_enhanced_list = enhanced_list[u: u+group_num]
                        sub_enhanced_index_list = [enhanced_text_to_features_index[enhanced_text] for enhanced_text in sub_enhanced_list]
                        sub_enhanced_text_features = enhanced_all_text_features[sub_enhanced_index_list]

                        sub_enhanced_text_features = torch.cat([sub_enhanced_text_features, raw_text_features[index].view(1, -1)], 0)

                        sub_enhanced_text_features /= sub_enhanced_text_features.norm(dim=-1, keepdim=True)
                        similarity = (100.0 * sub_enhanced_text_features @ image_features.T).softmax(dim=-1)

                        first_reacall_img_index = []
                        for enhance_i in range(sub_enhanced_text_features.shape[0]):
                            values, pre_indexes = similarity[enhance_i].topk(topk_num+12)
                            for values, j in zip(values, pre_indexes):
                                pre_id = test_img_ids[j]
                                first_reacall_img_index.append(j.item())
                        first_reacall_img_index = list(set(first_reacall_img_index))

                        sub_img_features = image_features[first_reacall_img_index]
                        similarity = (100.0 * sub_enhanced_text_features @ sub_img_features.T).softmax(dim=-1)


                        sub_counter = Counter()
                        for enhance_i in range(sub_enhanced_text_features.shape[0]):
                            values, pre_indexes = similarity[enhance_i].topk(topk_num+8)
                            # values, pre_indexes = similarity[enhance_i].topk(len(first_reacall_img_index))
                            pre_ids = [test_img_ids[first_reacall_img_index[j]] for j in pre_indexes]
                            sub_counter.update(pre_ids)
                        pre_ids = [img_id for (img_id, _) in sub_counter.most_common(topk_num+5)]
                        counter.update(pre_ids)
                    pre_ids = [img_id for (img_id, _) in counter.most_common(topk_num)]
                r = 0
                for true_id in true_ids:
                    if true_id in pre_ids:
                        r += 1
                recall_list.append(r / min(len(true_ids), topk_num))
            # type_recall_list[type_to_index[type]].append(r / min(len(true_ids), topk_num))
            # check
            # new_enhance_index_list = [n_i for n_i, enhance_text in enumerate(enhance_list) if text_enhance_to_flag[text][enhance_text] == 1] # 保留的enhance_text的索引
            #
            # index = get_text_index(text, sub_test_sen_to_ids)
            # enhance_text_features = torch.cat([text_features[index][new_enhance_index_list], raw_text_features[index].view(1, -1)], 0)

            recall = sum(recall_list) / len(recall_list)
            fload_recall_list.append(recall)
            print('\tflod [{flod_i}], recall [{recall}]'.format(
                flod_i=flod_i,
                model=name,
                recall=recall
            ))
    print('model [{model}], recall [{recall}]'.format(
        model=name,
        recall=sum(fload_recall_list) / len(fload_recall_list)
    ))

align-base
	flod [2], recall [0.7233968253968254]
	flod [7], recall [0.7463148148148148]
model [align-base], recall [0.7348558201058201]
clipseg-rd64-refined
	flod [2], recall [0.7024285714285716]
	flod [7], recall [0.6613518518518517]
model [clipseg-rd64-refined], recall [0.6818902116402117]
clip-vit-base-patch32
	flod [2], recall [0.6515793650793652]
	flod [7], recall [0.6815925925925926]
model [clip-vit-base-patch32], recall [0.6665859788359789]
groupvit-gcc-yfcc
	flod [2], recall [0.5061746031746033]
	flod [7], recall [0.585537037037037]
model [groupvit-gcc-yfcc], recall [0.5458558201058201]


## chatgpt v2 with not vote

In [45]:


def get_text_index(text, text_to_enhanced):
    index = list(text_to_enhanced.keys()).index(text)
    return index


sub_test_sen_to_ids = json.load(open('../combine/data/sub_test_sen_to_ids_300*5.json', 'r', encoding='utf-8'))
test_data_11 = json.load(open('data/test_data_11.json', 'r', encoding='utf-8'))
test_img_ids = json.load(open('../encoder/data/test_img_ids.json', 'r', encoding='utf-8'))

text_to_enhanced = json.load(open('../enhance/chatGPT/data/text_to_enhanced_list.json', 'r', encoding='utf-8'))

text_enhance_to_flag = json.load(open('../enhance/chatGPT/data/text_enhance_to_flag_sub.json', 'r', encoding='utf-8'))

topk_num = cfg['topk_num']

for name in model_name_list:
    fload_recall_list = []
    print(name)
    for flod_i, test_data in enumerate(test_data_11):
        if flod_i in [2,7]:
            test_data_index_list = [get_text_index(text, sub_test_sen_to_ids) for text in test_data]
            image_features = torch.load('../encoder/data/image_features_{}.pt'.format(name),
                                        map_location=torch.device('cpu'))
            image_features = torch.stack(image_features).view(1000, -1)

            text_features = torch.load('../enhance/chatGPT/data/enhanced_text_features_{}.pt'.format(name),map_location=torch.device('cpu')) # list

            raw_text_features = torch.load('../encoder/data/raw_text_features_{}.pt'.format(name), map_location=torch.device('cpu'))
            raw_text_features /= raw_text_features.norm(dim=-1, keepdim=True)


            image_features /= image_features.norm(dim=-1, keepdim=True)
            # 14-7
            recall_list = []
            for i, text in enumerate(test_data):
                enhance_list = text_to_enhanced[text]
                type = sub_test_sen_to_ids[text]['type']

                # check
                # new_enhance_index_list = [n_i for n_i, enhance_text in enumerate(enhance_list) if text_enhance_to_flag[text][enhance_text] == 1] # 保留的enhance_text的索引
                #
                # enhance_index = get_text_index(text, text_to_enhanced)
                # raw_index = get_text_index(text, sub_test_sen_to_ids)
                # enhance_text_features = torch.cat([text_features[enhance_index][new_enhance_index_list], raw_text_features[raw_index].view(1, -1)], 0)
                # enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)

                # no check
                enhance_index = get_text_index(text, text_to_enhanced)
                raw_index = get_text_index(text, sub_test_sen_to_ids)
                enhance_text_features = torch.cat([text_features[enhance_index], raw_text_features[raw_index].view(1, -1)], 0)
                enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)


                # if type in ['fragment', 'phrase']:
                #     index = get_text_index(text, text_to_enhanced)
                #     enhance_text_features = torch.cat([text_features[index], raw_text_features[index].view(1, -1)], 0)
                #     enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)
                # else:
                #    enhance_text_features = raw_text_features[index].view(1, -1)
                #    # enhance_text_features /= enhance_text_features.norm(dim=-1, keepdim=True)

                similarity = (100.0 * enhance_text_features @ image_features.T).softmax(dim=-1)

                true_ids = sub_test_sen_to_ids[text]['images']
                true_ids_new = [img_id for img_id in true_ids if img_id in test_img_ids]
                true_ids = true_ids_new

                img_to_sim = defaultdict(list)

                first_reacall_img_index = []
                for enhance_i in range(enhance_text_features.shape[0]):
                    values, pre_indexes = similarity[enhance_i].topk(topk_num + 12)
                    for values, j in zip(values, pre_indexes):
                        pre_id = test_img_ids[j]
                        first_reacall_img_index.append(j.item())
                first_reacall_img_index = list(set(first_reacall_img_index))

                sub_img_features = image_features[first_reacall_img_index]
                similarity = (100.0 * enhance_text_features @ sub_img_features.T).softmax(dim=-1)

                counter = Counter()
                for enhance_i in range(enhance_text_features.shape[0]):
                    values, pre_indexes = similarity[enhance_i].topk(topk_num + 8)
                    pre_ids = [test_img_ids[first_reacall_img_index[j]] for j in pre_indexes]
                    counter.update(pre_ids)
                pre_ids = [img_id for (img_id, _) in counter.most_common(topk_num)]

                r = 0
                for true_id in true_ids:
                    if true_id in pre_ids:
                        r += 1
                recall_list.append(r / min(len(true_ids), topk_num))

            recall = sum(recall_list) / len(recall_list)
            fload_recall_list.append(recall)
            print('\tflod [{flod_i}], recall [{recall}]'.format(
                flod_i=flod_i,
                model=name,
                recall=recall
            ))
    print('model [{model}], recall [{recall}]'.format(
        model=name,
        recall=sum(fload_recall_list) / len(fload_recall_list)
    ))

align-base
	flod [2], recall [0.7302222222222223]
	flod [7], recall [0.7336296296296295]
model [align-base], recall [0.7319259259259259]
clipseg-rd64-refined
	flod [2], recall [0.7035793650793652]
	flod [7], recall [0.6584444444444444]
model [clipseg-rd64-refined], recall [0.6810119047619048]
clip-vit-base-patch32
	flod [2], recall [0.6406190476190476]
	flod [7], recall [0.6738703703703706]
model [clip-vit-base-patch32], recall [0.6572447089947091]
groupvit-gcc-yfcc
	flod [2], recall [0.5105000000000002]
	flod [7], recall [0.5660925925925926]
model [groupvit-gcc-yfcc], recall [0.5382962962962964]
